<a href="https://colab.research.google.com/github/Saikadam123/ADM-Project/blob/main/IO_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# How to Use

Step 1: Run All
Step 2: Connect to Google Drive for access to Dataset
Step 3: Search for Image file in the Interface

In [ ]:
!pip install -q -U transformers huggingface_hub safetensors ipywidgets scikit-learn

In [ ]:
# Set up dependencies, mount storage, establish device and reproducibility settings, and configure paths and hyperparameters for multimodal CBP model inference and explainability.

import os, io, json, random, warnings
from contextlib import nullcontext

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, transforms
from transformers import AutoModel, AutoTokenizer

import matplotlib.pyplot as plt
from IPython.display import display, clear_output
import ipywidgets as widgets

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except Exception as exc:
    print("Google Drive mount skipped:", exc)

DATASET_DIR = os.environ.get("CBP_DATASET_DIR", "/content/drive/MyDrive/NewDataset")
IMAGE_FOLDER = os.path.join(DATASET_DIR, "images_normalized")
METADATA_CSV_PATH = os.path.join(DATASET_DIR, "multimodal_dataset_no_label.csv")
MODEL_FOLDER = os.path.join(DATASET_DIR, "trained_model_robustness_baseline")
BEST_MODEL_PATH = os.path.join(MODEL_FOLDER, "best_model.pth")

ROBUSTNESS_SUMMARY_PATH = os.path.join(MODEL_FOLDER, "robustness_summary.json")
VALIDATION_RESULTS_PATH = os.path.join(MODEL_FOLDER, "validation_results.csv")

TEXT_MODEL_NAME = "emilyalsentzer/Bio_ClinicalBERT"
MAX_TEXT_LEN = 256
IMAGE_SIZE = 224
DENSENET_IMAGE_MEAN = [0.485, 0.456, 0.406]
DENSENET_IMAGE_STD = [0.229, 0.224, 0.225]
DENSENET_FEATURE_DIM = 1024
TEXT_DIM = 768
IMAGE_DIM = 768
CBP_PROJ_DIM = 192
CBP_CONV_FILTERS = 4
FUSION_DIM = 256
CLASSIFIER_DROPOUT = 0.3

OCCLUSION_PATCH_SIZE = 32
OCCLUSION_STRIDE = 16
TOKEN_PERTURB_BATCH = 24
IMAGE_PERTURB_BATCH = 16
MAX_EXPLAIN_TOKENS = 96
FAITHFULNESS_FRACTIONS = np.linspace(0.0, 1.0, 11)
INFERENCE_SUBSTITUTE = "zero"
INFERENCE_NOISE_SIGMA = 0.0

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_ENABLED = torch.cuda.is_available()

def amp_autocast():
    return torch.autocast(device_type="cuda", dtype=torch.float16) if AMP_ENABLED else nullcontext()

print("Device:", device)
print("Dataset:", DATASET_DIR)
print("Metadata:", METADATA_CSV_PATH)
print("Checkpoint:", BEST_MODEL_PATH)


In [ ]:
# Validate essential file paths and metadata schema, ensure unique image references, and load the tuned validation decision threshold with a fallback.

required_paths = {
    "metadata CSV": METADATA_CSV_PATH,
    "trained checkpoint": BEST_MODEL_PATH,
}
missing = [f"{name}: {path}" for name, path in required_paths.items() if not os.path.exists(path)]
if missing:
    raise FileNotFoundError(
        "Required file(s) not found. Check Drive mount / DATASET_DIR:\n  - " + "\n  - ".join(missing)
    )

metadata_df = pd.read_csv(METADATA_CSV_PATH)
required_cols = {"uid", "filename", "findings"}
HAS_LABEL = "binary_label" in metadata_df.columns
missing_cols = required_cols.difference(metadata_df.columns)
if missing_cols:
    raise ValueError(f"Metadata CSV missing columns: {sorted(missing_cols)}")

metadata_df["filename_key"] = metadata_df["filename"].astype(str).map(
    lambda x: os.path.basename(x).strip().lower()
)
if metadata_df["filename_key"].duplicated().any():
    dupes = metadata_df.loc[
        metadata_df["filename_key"].duplicated(keep=False), "filename"
    ].head(10).tolist()
    raise ValueError(f"Filename lookup is not unique. Examples: {dupes}")

def load_validation_threshold():
    if os.path.exists(ROBUSTNESS_SUMMARY_PATH):
        with open(ROBUSTNESS_SUMMARY_PATH, "r", encoding="utf-8") as f:
            summary = json.load(f)
        if "validation_selected_threshold" in summary:
            return float(summary["validation_selected_threshold"]), "robustness_summary.json"

    if os.path.exists(VALIDATION_RESULTS_PATH):
        vdf = pd.read_csv(VALIDATION_RESULTS_PATH)
        if "Threshold" in vdf.columns and vdf["Threshold"].notna().any():
            return float(vdf["Threshold"].dropna().iloc[0]), "validation_results.csv"

    warnings.warn(
        "No saved validation threshold found; using 0.5. "
        "Keep robustness_summary.json or validation_results.csv for exact reproduction."
    )
    return 0.5, "fallback_0.5"

BEST_THRESHOLD, THRESHOLD_SOURCE = load_validation_threshold()
print(f"Metadata rows: {len(metadata_df):,}")
print(f"Threshold: {BEST_THRESHOLD:.4f} ({THRESHOLD_SOURCE})")


In [ ]:
# Define model components and architecture for multimodal CBP classification, supporting attention pooling, latent noise injection, and flexible modality masking.

def apply_relative_gaussian_noise(x, sigma):
    if sigma is None or sigma <= 0:
        return x
    scale = x.detach().std(dim=-1, keepdim=True).clamp_min(1e-6)
    return x + sigma * scale * torch.randn_like(x)

class AttentionPooling(nn.Module):
    def __init__(self, hidden_dim, dropout=0.1):
        super().__init__()
        self.attn = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.Tanh(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1),
        )

    def forward(self, hidden_states, attention_mask=None):
        scores = self.attn(hidden_states).squeeze(-1)
        if attention_mask is not None:
            scores = scores.masked_fill(attention_mask == 0, float("-inf"))
        weights = torch.softmax(scores, dim=1).unsqueeze(-1)
        return (hidden_states * weights).sum(dim=1), weights.squeeze(-1)

class CompactBilinearPoolingCNN(nn.Module):
    def __init__(self, text_dim=TEXT_DIM, image_dim=IMAGE_DIM, proj_dim=CBP_PROJ_DIM,
                 conv_filters=CBP_CONV_FILTERS, fusion_dim=FUSION_DIM,
                 dropout=CLASSIFIER_DROPOUT):
        super().__init__()
        self.proj_dim = proj_dim
        self.text_proj = nn.Linear(text_dim, proj_dim)
        self.image_proj = nn.Linear(image_dim, proj_dim)
        self.proj_dropout = nn.Dropout(dropout)
        self.conv_block = nn.Sequential(
            nn.Conv2d(1, conv_filters, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2),
        )
        pooled_dim = proj_dim // 2
        conv_out_dim = conv_filters * pooled_dim * pooled_dim
        self.fusion_proj = nn.Sequential(
            nn.Linear(conv_out_dim + proj_dim + proj_dim, fusion_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
        )

    def forward(self, text_features, image_features):
        t = self.proj_dropout(self.text_proj(text_features))
        v = self.proj_dropout(self.image_proj(image_features))
        bilinear_map = torch.bmm(v.unsqueeze(2), t.unsqueeze(1)).unsqueeze(1)
        compressed = self.conv_block(bilinear_map).flatten(start_dim=1)
        return self.fusion_proj(torch.cat([compressed, t, v], dim=1))

class MultimodalCBPClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.image_encoder = models.densenet121(weights=None)
        self.image_encoder.classifier = nn.Identity()
        self.text_encoder = AutoModel.from_pretrained(TEXT_MODEL_NAME)

        self.text_attn_pool = AttentionPooling(self.text_encoder.config.hidden_size)
        self.image_attn_pool = AttentionPooling(DENSENET_FEATURE_DIM)
        self.image_projection = nn.Sequential(
            nn.LayerNorm(DENSENET_FEATURE_DIM),
            nn.Linear(DENSENET_FEATURE_DIM, IMAGE_DIM),
        )
        self.cbp = CompactBilinearPoolingCNN()
        self.classifier = nn.Sequential(
            nn.Linear(FUSION_DIM, FUSION_DIM // 2),
            nn.ReLU(inplace=True),
            nn.Dropout(CLASSIFIER_DROPOUT),
            nn.Linear(FUSION_DIM // 2, 1),
        )
        self.register_buffer("text_missing", torch.zeros(TEXT_DIM))
        self.register_buffer("image_missing", torch.zeros(IMAGE_DIM))

    def encode_image(self, pixel_values):
        image_map = F.relu(self.image_encoder.features(pixel_values), inplace=False)
        image_tokens = image_map.flatten(2).transpose(1, 2).contiguous()
        image_pooled, _ = self.image_attn_pool(image_tokens, attention_mask=None)
        return self.image_projection(image_pooled)

    def encode(self, pixel_values, input_ids, attention_mask):
        image_features = self.encode_image(pixel_values)
        text_out = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask)
        text_features, _ = self.text_attn_pool(
            text_out.last_hidden_state, attention_mask=attention_mask
        )
        return text_features, image_features

    def fuse_and_classify(self, text_features, image_features, modality_mask=None,
                          noise_sigma=0.0, substitute="zero", return_extras=False):
        if substitute not in ("zero", "calibrated"):
            raise ValueError("substitute must be 'zero' or 'calibrated'")
        b = text_features.shape[0]
        if modality_mask is None:
            modality_mask = torch.ones(
                b, 2, device=text_features.device, dtype=text_features.dtype
            )
        modality_mask = modality_mask.to(
            device=text_features.device, dtype=text_features.dtype
        )

        if substitute == "zero":
            t_sub = torch.zeros_like(text_features)
            v_sub = torch.zeros_like(image_features)
        else:
            t_sub = self.text_missing.to(text_features.dtype).unsqueeze(0).expand_as(text_features)
            v_sub = self.image_missing.to(image_features.dtype).unsqueeze(0).expand_as(image_features)

        mt, mv = modality_mask[:, 0:1], modality_mask[:, 1:2]
        t = mt * text_features + (1.0 - mt) * t_sub
        v = mv * image_features + (1.0 - mv) * v_sub
        t = apply_relative_gaussian_noise(t, noise_sigma)
        v = apply_relative_gaussian_noise(v, noise_sigma)

        fused = self.cbp(t, v)
        logits = self.classifier(fused).squeeze(-1)
        if return_extras:
            return logits, {"fused": fused, "text": t, "image": v}
        return logits

    def forward(self, pixel_values, input_ids, attention_mask, modality_mask=None,
                noise_sigma=0.0, substitute="zero"):
        text_features, image_features = self.encode(
            pixel_values, input_ids, attention_mask
        )
        return self.fuse_and_classify(
            text_features, image_features, modality_mask, noise_sigma, substitute
        )

def modality_mask(batch_size, condition, device_, dtype=torch.float32):
    masks = {
        "multimodal": (1.0, 1.0),
        "text_only": (1.0, 0.0),
        "image_only": (0.0, 1.0),
    }
    if condition not in masks:
        raise ValueError(f"Unknown condition: {condition}")
    mt, mv = masks[condition]
    return torch.tensor([[mt, mv]], device=device_, dtype=dtype).expand(batch_size, 2)


In [ ]:
# Initialize data preprocessors, instantiate the multimodal CBP model, and strictly load the trained model weights from the checkpoint.

class DenseNetImageProcessor:
    def __init__(self, image_size=IMAGE_SIZE):
        self.transform = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=DENSENET_IMAGE_MEAN, std=DENSENET_IMAGE_STD),
        ])
    def __call__(self, image):
        return self.transform(image.convert("RGB")).unsqueeze(0)

tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL_NAME)
if tokenizer.mask_token_id is None:
    raise ValueError("Tokenizer has no mask token; word occlusion cannot run.")
image_processor = DenseNetImageProcessor()

model = MultimodalCBPClassifier().to(device)

try:
    state = torch.load(BEST_MODEL_PATH, map_location="cpu", weights_only=True)
except TypeError:
    state = torch.load(BEST_MODEL_PATH, map_location="cpu")

if isinstance(state, dict) and "state_dict" in state and isinstance(state["state_dict"], dict):
    state = state["state_dict"]
elif isinstance(state, dict) and "model_state_dict" in state and isinstance(state["model_state_dict"], dict):
    state = state["model_state_dict"]

if not isinstance(state, dict):
    raise TypeError("Checkpoint is not a state_dict-compatible mapping.")
if state and all(str(k).startswith("module.") for k in state):
    state = {str(k)[7:]: v for k, v in state.items()}

model.load_state_dict(state, strict=True)
model.eval()
print("Checkpoint loaded strictly.")


In [ ]:
# Helper functions for input preprocessing, condition-specific model inference, thresholded prediction, and metadata filename lookup.

def encode_text(findings_text):
    encoded = tokenizer(
        str(findings_text),
        padding="max_length",
        truncation=True,
        max_length=MAX_TEXT_LEN,
        return_tensors="pt",
    )
    return encoded["input_ids"].to(device), encoded["attention_mask"].to(device)

def prepare_inputs(image, findings_text=""):
    return (image_processor(image).to(device), *encode_text(findings_text))

@torch.inference_mode()
def forward_prob_abnormal(pixel_values, input_ids, attention_mask, condition="multimodal"):
    mask = modality_mask(
        pixel_values.shape[0], condition, pixel_values.device, pixel_values.dtype
    )
    with amp_autocast():
        logits = model(
            pixel_values=pixel_values,
            input_ids=input_ids,
            attention_mask=attention_mask,
            modality_mask=mask,
            noise_sigma=INFERENCE_NOISE_SIGMA,
            substitute=INFERENCE_SUBSTITUTE,
        )
    return torch.sigmoid(logits.float())

def target_confidence(prob_abnormal, target_class):
    p = float(prob_abnormal)
    return p if int(target_class) == 1 else 1.0 - p

@torch.inference_mode()
def infer_single(pixel_values, input_ids, attention_mask, condition):
    p = float(
        forward_prob_abnormal(pixel_values, input_ids, attention_mask, condition)
        .squeeze().cpu()
    )
    pred = int(p >= BEST_THRESHOLD)
    return {
        "prob_abnormal": p,
        "prob_normal": 1.0 - p,
        "predicted_label": pred,
        "predicted_class": "ABNORMAL" if pred else "NORMAL",
        "threshold": float(BEST_THRESHOLD),
        "condition": condition,
    }

def lookup_metadata(uploaded_filename):
    key = os.path.basename(str(uploaded_filename)).strip().lower()
    hit = metadata_df.loc[metadata_df["filename_key"] == key]
    if len(hit) == 0:
        return None
    if len(hit) != 1:
        raise RuntimeError(f"Expected one metadata row; found {len(hit)}.")
    return hit.iloc[0]


In [ ]:
# Normalize patient UIDs, locate image files on disk recursively, and build a UID/filename lookup index for dataset queries.

def _norm_uid(x):
    if isinstance(x, (int, np.integer)):
        return str(int(x))
    if isinstance(x, (float, np.floating)) and float(x).is_integer():
        return str(int(x))
    return str(x).strip()

metadata_df["uid_key"] = metadata_df["uid"].map(_norm_uid)

def resolve_image_path(filename):
    base = os.path.basename(str(filename)).strip()
    direct = os.path.join(IMAGE_FOLDER, base)
    if os.path.exists(direct):
        return direct
    target = base.lower()
    for root, _, files in os.walk(IMAGE_FOLDER):
        for f in files:
            if f.lower() == target:
                return os.path.join(root, f)
    raise FileNotFoundError(f"Image '{base}' not found under {IMAGE_FOLDER}")

def lookup_by_uid(uid, filename=None):
    key = _norm_uid(uid)
    hit = metadata_df.loc[metadata_df["uid_key"] == key]
    if len(hit) == 0:
        raise KeyError(f"UID {key} not found in {os.path.basename(METADATA_CSV_PATH)}")
    if len(hit) > 1:
        fkey = os.path.basename(str(filename or "")).strip().lower()
        if fkey:
            hit = hit.loc[hit["filename_key"] == fkey]
        if len(hit) != 1:
            raise RuntimeError(
                f"UID {key} matches several images: {metadata_df.loc[metadata_df['uid_key'] == key, 'filename'].tolist()}. "
                "Enter the exact filename to choose one."
            )
    return hit.iloc[0]

print("UID index built. Unique UIDs:", metadata_df["uid_key"].nunique(), "of", len(metadata_df), "rows")


In [ ]:
# Implement occlusion-based explainability routines to measure feature importance, generating spatial image heatmaps and ranked text-token attributions.

def content_word_groups(input_ids_1d, attention_mask_1d):
    special_ids = set(tokenizer.all_special_ids)
    ids = input_ids_1d.detach().cpu().tolist()
    mask = attention_mask_1d.detach().cpu().tolist()
    valid_positions = [
        i for i, (tok_id, attended) in enumerate(zip(ids, mask))
        if int(attended) == 1 and int(tok_id) not in special_ids
    ][:MAX_EXPLAIN_TOKENS]

    groups = []
    for pos in valid_positions:
        tok = tokenizer.convert_ids_to_tokens(int(ids[pos]))
        if tok.startswith("##") and groups:
            groups[-1]["word"] += tok[2:]
            groups[-1]["positions"].append(int(pos))
        else:
            groups.append({"word": tok, "positions": [int(pos)]})
    return groups

def patch_coordinates(height, width, patch_size=OCCLUSION_PATCH_SIZE,
                      stride=OCCLUSION_STRIDE):
    if patch_size > height or patch_size > width:
        raise ValueError("patch_size exceeds image dimensions")
    ys = list(range(0, height - patch_size + 1, stride))
    xs = list(range(0, width - patch_size + 1, stride))
    if ys[-1] != height - patch_size:
        ys.append(height - patch_size)
    if xs[-1] != width - patch_size:
        xs.append(width - patch_size)
    return [(y, y + patch_size, x, x + patch_size) for y in ys for x in xs]

def auc_over_fraction(fractions, confidences):
    x = np.asarray(fractions, dtype=float)
    y = np.asarray(confidences, dtype=float)
    trap = getattr(np, "trapezoid", None)
    return float(trap(y, x) if trap is not None else np.trapz(y, x))

def denormalize_image(pixel_values_1):
    x = pixel_values_1[0].detach().float().cpu()
    mean_t = torch.tensor(DENSENET_IMAGE_MEAN, dtype=x.dtype).view(-1, 1, 1)
    std_t = torch.tensor(DENSENET_IMAGE_STD, dtype=x.dtype).view(-1, 1, 1)
    x = (x * std_t + mean_t).clamp(0, 1)
    return x.permute(1, 2, 0).numpy()

@torch.inference_mode()
def explain_image_patches(pixel_values, input_ids, attention_mask, target_class,
                          condition, patch_size=OCCLUSION_PATCH_SIZE,
                          stride=OCCLUSION_STRIDE,
                          perturb_batch_size=IMAGE_PERTURB_BATCH):
    _, _, h, w = pixel_values.shape
    base_p = float(
        forward_prob_abnormal(pixel_values, input_ids, attention_mask, condition)
        .squeeze().cpu()
    )
    base_conf = target_confidence(base_p, target_class)
    coords = patch_coordinates(h, w, patch_size, stride)

    rows = []
    heat_sum = np.zeros((h, w), dtype=np.float32)
    heat_count = np.zeros((h, w), dtype=np.float32)

    for start in range(0, len(coords), perturb_batch_size):
        chunk = coords[start:start + perturb_batch_size]
        b = len(chunk)
        images = pixel_values.repeat(b, 1, 1, 1)
        ids = input_ids.repeat(b, 1)
        masks = attention_mask.repeat(b, 1)
        for j, (y0, y1, x0, x1) in enumerate(chunk):
            images[j, :, y0:y1, x0:x1] = 0.0
        probs = forward_prob_abnormal(images, ids, masks, condition).cpu().numpy().ravel()

        for coord, p in zip(chunk, probs):
            y0, y1, x0, x1 = coord
            occ_conf = target_confidence(float(p), target_class)
            score = base_conf - occ_conf
            rows.append({
                "y0": y0, "y1": y1, "x0": x0, "x1": x1,
                "base_confidence": base_conf,
                "occluded_confidence": occ_conf,
                "importance": score,
            })
            heat_sum[y0:y1, x0:x1] += score
            heat_count[y0:y1, x0:x1] += 1.0

    patch_df = pd.DataFrame(rows).sort_values(
        "importance", ascending=False
    ).reset_index(drop=True)
    heatmap = heat_sum / np.maximum(heat_count, 1.0)
    return patch_df, heatmap

@torch.inference_mode()
def explain_text_words(pixel_values, input_ids, attention_mask, target_class,
                       perturb_batch_size=TOKEN_PERTURB_BATCH):
    groups = content_word_groups(input_ids[0], attention_mask[0])
    if not groups:
        return pd.DataFrame(columns=[
            "word", "positions", "base_confidence", "occluded_confidence", "importance"
        ])

    base_p = float(
        forward_prob_abnormal(pixel_values, input_ids, attention_mask, "multimodal")
        .squeeze().cpu()
    )
    base_conf = target_confidence(base_p, target_class)
    rows = []

    for start in range(0, len(groups), perturb_batch_size):
        chunk = groups[start:start + perturb_batch_size]
        b = len(chunk)
        ids = input_ids.repeat(b, 1)
        masks = attention_mask.repeat(b, 1)
        images = pixel_values.repeat(b, 1, 1, 1)
        for j, group in enumerate(chunk):
            ids[j, group["positions"]] = tokenizer.mask_token_id
        probs = forward_prob_abnormal(
            images, ids, masks, "multimodal"
        ).cpu().numpy().ravel()
        for group, p in zip(chunk, probs):
            occ_conf = target_confidence(float(p), target_class)
            rows.append({
                "word": group["word"],
                "positions": group["positions"],
                "base_confidence": base_conf,
                "occluded_confidence": occ_conf,
                "importance": base_conf - occ_conf,
            })
    return pd.DataFrame(rows).sort_values(
        "importance", ascending=False
    ).reset_index(drop=True)


In [ ]:
# Evaluate modality-level attributions and calculate deletion/insertion faithfulness curves and AUC metrics across image, text, and joint features.

@torch.inference_mode()
def modality_level_importance(pixel_values, input_ids, attention_mask, target_class):
    with amp_autocast():
        t, v = model.encode(pixel_values, input_ids, attention_mask)
    probs = {}
    for condition in ("multimodal", "text_only", "image_only"):
        m = modality_mask(t.shape[0], condition, t.device, t.dtype)
        with amp_autocast():
            logits = model.fuse_and_classify(
                t, v, m, INFERENCE_NOISE_SIGMA, INFERENCE_SUBSTITUTE
            )
        probs[condition] = float(torch.sigmoid(logits.float()).squeeze().cpu())

    conf = lambda p: target_confidence(p, target_class)
    full = conf(probs["multimodal"])
    text_only = conf(probs["text_only"])
    image_only = conf(probs["image_only"])
    return {
        "full_target_confidence": full,
        "image_removed_target_confidence": text_only,
        "text_removed_target_confidence": image_only,
        "image_modality_importance": full - text_only,
        "text_modality_importance": full - image_only,
        "p_abnormal_multimodal": probs["multimodal"],
        "p_abnormal_text_only": probs["text_only"],
        "p_abnormal_image_only": probs["image_only"],
    }

@torch.inference_mode()
def confidence_for_inputs(pixel_values, input_ids, attention_mask, target_class, condition):
    p = float(
        forward_prob_abnormal(pixel_values, input_ids, attention_mask, condition)
        .squeeze().cpu()
    )
    return target_confidence(p, target_class)

@torch.inference_mode()
def image_faithfulness_curves(pixel_values, input_ids, attention_mask, target_class,
                              condition, patch_df, fractions=FAITHFULNESS_FRACTIONS):
    ranked = patch_df[patch_df["importance"] > 0].copy()
    if len(ranked) == 0:
        ranked = patch_df.copy()
    coords = list(ranked[["y0","y1","x0","x1"]].astype(int).itertuples(
        index=False, name=None
    ))

    deletion, insertion = [], []
    for frac in fractions:
        k = int(round(float(frac) * len(coords)))
        chosen = coords[:k]

        dimg = pixel_values.clone()
        for y0, y1, x0, x1 in chosen:
            dimg[:, :, y0:y1, x0:x1] = 0.0
        deletion.append(confidence_for_inputs(
            dimg, input_ids, attention_mask, target_class, condition
        ))

        iimg = torch.zeros_like(pixel_values)
        for y0, y1, x0, x1 in chosen:
            iimg[:, :, y0:y1, x0:x1] = pixel_values[:, :, y0:y1, x0:x1]
        insertion.append(confidence_for_inputs(
            iimg, input_ids, attention_mask, target_class, condition
        ))

    return {
        "fractions": np.asarray(fractions),
        "deletion_confidence": np.asarray(deletion),
        "insertion_confidence": np.asarray(insertion),
        "deletion_auc": auc_over_fraction(fractions, deletion),
        "insertion_auc": auc_over_fraction(fractions, insertion),
    }

@torch.inference_mode()
def text_faithfulness_curves(pixel_values, input_ids, attention_mask, target_class,
                             word_df, fractions=FAITHFULNESS_FRACTIONS):
    ranked = word_df[word_df["importance"] > 0]
    if len(ranked) == 0:
        ranked = word_df
    ranked_groups = ranked["positions"].tolist()
    all_positions = [
        p for group in content_word_groups(input_ids[0], attention_mask[0])
        for p in group["positions"]
    ]

    deletion, insertion = [], []
    for frac in fractions:
        k = int(round(float(frac) * len(ranked_groups)))
        chosen = [p for group in ranked_groups[:k] for p in group]

        dids = input_ids.clone()
        if chosen:
            dids[0, chosen] = tokenizer.mask_token_id
        deletion.append(confidence_for_inputs(
            pixel_values, dids, attention_mask, target_class, "multimodal"
        ))

        iids = input_ids.clone()
        if all_positions:
            iids[0, all_positions] = tokenizer.mask_token_id
        if chosen:
            iids[0, chosen] = input_ids[0, chosen]
        insertion.append(confidence_for_inputs(
            pixel_values, iids, attention_mask, target_class, "multimodal"
        ))

    return {
        "fractions": np.asarray(fractions),
        "deletion_confidence": np.asarray(deletion),
        "insertion_confidence": np.asarray(insertion),
        "deletion_auc": auc_over_fraction(fractions, deletion),
        "insertion_auc": auc_over_fraction(fractions, insertion),
    }

@torch.inference_mode()
def joint_faithfulness_curves(pixel_values, input_ids, attention_mask, target_class,
                              word_df, patch_df, fractions=FAITHFULNESS_FRACTIONS):
    units = []
    for row in word_df.itertuples(index=False):
        units.append({
            "kind": "word", "importance": float(row.importance),
            "positions": list(row.positions)
        })
    for row in patch_df.itertuples(index=False):
        units.append({
            "kind": "patch", "importance": float(row.importance),
            "coord": (int(row.y0), int(row.y1), int(row.x0), int(row.x1))
        })

    positive = [u for u in units if u["importance"] > 0]
    ranked = sorted(positive if positive else units,
                    key=lambda u: u["importance"], reverse=True)
    all_positions = [
        p for group in content_word_groups(input_ids[0], attention_mask[0])
        for p in group["positions"]
    ]

    deletion, insertion = [], []
    for frac in fractions:
        k = int(round(float(frac) * len(ranked)))
        chosen = ranked[:k]

        dimg, dids = pixel_values.clone(), input_ids.clone()
        for unit in chosen:
            if unit["kind"] == "word":
                dids[0, unit["positions"]] = tokenizer.mask_token_id
            else:
                y0, y1, x0, x1 = unit["coord"]
                dimg[:, :, y0:y1, x0:x1] = 0.0
        deletion.append(confidence_for_inputs(
            dimg, dids, attention_mask, target_class, "multimodal"
        ))

        iimg, iids = torch.zeros_like(pixel_values), input_ids.clone()
        if all_positions:
            iids[0, all_positions] = tokenizer.mask_token_id
        for unit in chosen:
            if unit["kind"] == "word":
                pos = unit["positions"]
                iids[0, pos] = input_ids[0, pos]
            else:
                y0, y1, x0, x1 = unit["coord"]
                iimg[:, :, y0:y1, x0:x1] = pixel_values[:, :, y0:y1, x0:x1]
        insertion.append(confidence_for_inputs(
            iimg, iids, attention_mask, target_class, "multimodal"
        ))

    return {
        "fractions": np.asarray(fractions),
        "deletion_confidence": np.asarray(deletion),
        "insertion_confidence": np.asarray(insertion),
        "deletion_auc": auc_over_fraction(fractions, deletion),
        "insertion_auc": auc_over_fraction(fractions, insertion),
    }


In [ ]:
# Compute end-to-end multimodal predictions, run occlusion-based explainability with faithfulness metrics, and visualize results via plots and summary tables.

def top20_confidence_drop(curves):
    idx = int(np.argmin(np.abs(curves["fractions"] - 0.20)))
    return float(curves["deletion_confidence"][0] - curves["deletion_confidence"][idx])

def explain_one_image(image, uploaded_filename, manual_findings=""):
    row = lookup_metadata(uploaded_filename)
    manual_findings = str(manual_findings or "").strip()

    if manual_findings:
        findings_text = manual_findings
        text_source = "manual"
        condition = "multimodal"
    elif row is not None:
        findings_text = str(row["findings"])
        text_source = "metadata_filename_match"
        condition = "multimodal"
    else:
        findings_text = ""
        text_source = "none_image_only"
        condition = "image_only"

    pixel_values, input_ids, attention_mask = prepare_inputs(image, findings_text)
    prediction = infer_single(pixel_values, input_ids, attention_mask, condition)
    target_class = int(prediction["predicted_label"])
    target_conf = target_confidence(prediction["prob_abnormal"], target_class)

    patch_df, heatmap = explain_image_patches(
        pixel_values, input_ids, attention_mask, target_class, condition
    )
    image_curves = image_faithfulness_curves(
        pixel_values, input_ids, attention_mask, target_class, condition, patch_df
    )

    word_df = pd.DataFrame()
    text_curves = joint_curves = modality_importance = None

    if condition == "multimodal":
        word_df = explain_text_words(
            pixel_values, input_ids, attention_mask, target_class
        )
        modality_importance = modality_level_importance(
            pixel_values, input_ids, attention_mask, target_class
        )
        if len(word_df) > 0:
            text_curves = text_faithfulness_curves(
                pixel_values, input_ids, attention_mask, target_class, word_df
            )
            joint_curves = joint_faithfulness_curves(
                pixel_values, input_ids, attention_mask, target_class, word_df, patch_df
            )

    metadata_record = None
    if row is not None:
        metadata_record = {
            "uid": row["uid"],
            "filename": row["filename"],
            "binary_label": (int(row["binary_label"]) if HAS_LABEL and pd.notna(row["binary_label"]) else None),
            "findings": str(row["findings"]),
        }

    return {
        "uploaded_filename": uploaded_filename,
        "text_source": text_source,
        "findings_used": findings_text,
        "metadata": metadata_record,
        "prediction": prediction,
        "target_class": target_class,
        "target_confidence": target_conf,
        "display_image": denormalize_image(pixel_values),
        "word_df": word_df,
        "patch_df": patch_df,
        "image_heatmap": heatmap,
        "modality_importance": modality_importance,
        "image_curves": image_curves,
        "text_curves": text_curves,
        "joint_curves": joint_curves,
        "faithfulness": {
            "image_deletion_auc": image_curves["deletion_auc"],
            "image_insertion_auc": image_curves["insertion_auc"],
            "image_top20_confidence_drop": top20_confidence_drop(image_curves),
            "text_deletion_auc": None if text_curves is None else text_curves["deletion_auc"],
            "text_insertion_auc": None if text_curves is None else text_curves["insertion_auc"],
            "joint_deletion_auc": None if joint_curves is None else joint_curves["deletion_auc"],
            "joint_insertion_auc": None if joint_curves is None else joint_curves["insertion_auc"],
            "joint_top20_confidence_drop": None if joint_curves is None else top20_confidence_drop(joint_curves),
        },
    }

def plot_result(result):
    rgb, hm = result["display_image"], result["image_heatmap"]
    word_df = result["word_df"]
    ic = result["image_curves"]
    tc, jc = result["text_curves"], result["joint_curves"]

    fig, axes = plt.subplots(2, 2, figsize=(15, 11))
    pred = result["prediction"]

    axes[0,0].imshow(rgb)
    axes[0,0].set_title(
        f"{pred['predicted_class']} | P(abnormal)={pred['prob_abnormal']:.3f}\n"
        f"mode={pred['condition']}"
    )
    axes[0,0].axis("off")

    axes[0,1].imshow(rgb)
    max_abs = float(np.max(np.abs(hm))) if hm.size else 0.0
    if max_abs > 0:
        overlay = axes[0,1].imshow(
            hm / (max_abs + 1e-12), cmap="coolwarm",
            alpha=0.50, vmin=-1, vmax=1
        )
        fig.colorbar(overlay, ax=axes[0,1], fraction=0.046, pad=0.04,
                     label="Target-confidence change")
    axes[0,1].set_title("Signed patch occlusion: red=support, blue=opposition")
    axes[0,1].axis("off")

    if len(word_df) > 0:
        wd = word_df.head(12).iloc[::-1]
        axes[1,0].barh(wd["word"], wd["importance"])
        axes[1,0].axvline(0, linewidth=1)
        axes[1,0].set_title("Most influential report words")
        axes[1,0].set_xlabel("Confidence drop when masked")
    else:
        axes[1,0].text(0.5, 0.5, "No text attribution in image-only mode",
                       ha="center", va="center")
        axes[1,0].axis("off")

    ax = axes[1,1]
    ax.plot(ic["fractions"], ic["deletion_confidence"], marker="o",
            label=f"Image deletion AUC={ic['deletion_auc']:.3f}")
    ax.plot(ic["fractions"], ic["insertion_confidence"], marker="o",
            label=f"Image insertion AUC={ic['insertion_auc']:.3f}")
    if tc is not None:
        ax.plot(tc["fractions"], tc["deletion_confidence"], linestyle=":",
                label=f"Text deletion AUC={tc['deletion_auc']:.3f}")
    if jc is not None:
        ax.plot(jc["fractions"], jc["deletion_confidence"], linestyle="--",
                label=f"Joint deletion AUC={jc['deletion_auc']:.3f}")
        ax.plot(jc["fractions"], jc["insertion_confidence"], linestyle="--",
                label=f"Joint insertion AUC={jc['insertion_auc']:.3f}")
    ax.set_ylim(0,1)
    ax.set_xlabel("Fraction of ranked features perturbed")
    ax.set_ylabel("Confidence in explained class")
    ax.set_title("Perturbation faithfulness")
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

def display_result(result):
    p = result["prediction"]
    print("="*88)
    print("CLASSIFICATION")
    print("="*88)
    print(f"Image          : {result['uploaded_filename']}")
    print(f"Inference mode : {p['condition']}")
    print(f"Text source    : {result['text_source']}")
    print(f"P(abnormal)    : {p['prob_abnormal']:.6f}")
    print(f"P(normal)      : {p['prob_normal']:.6f}")
    print(f"Threshold      : {p['threshold']:.4f} ({THRESHOLD_SOURCE})")
    print(f"Prediction     : {p['predicted_class']} ({p['predicted_label']})")
    print(f"Target conf.   : {result['target_confidence']:.6f}")

    if result["metadata"] is not None:
        md = result["metadata"]
        print(f"Matched UID    : {md['uid']}")
        if md.get('binary_label') is not None:
            gt = int(md['binary_label'])
            print(f"Dataset label  : {gt} = {'ABNORMAL' if gt else 'NORMAL'} (reference only; not a model input)")
            print(f"Agreement      : {'MATCH' if gt == p['predicted_label'] else 'MISMATCH'} (model prediction vs dataset label)")
        else:
            print("Dataset label  : not available (CSV has no binary_label column)")
    else:
        print("Metadata match : none")

    if p["condition"] == "image_only":
        print("\nNOTE: no matched report/manual findings; this is native IMAGE-ONLY inference.")

    print("\nVERIFIABLE EXPLAINABILITY METRICS")
    for k, v in result["faithfulness"].items():
        if v is not None:
            print(f"{k:32s}: {v:.6f}")
    print("Lower deletion AUC / higher insertion AUC = stronger perturbation faithfulness.")

    if result["modality_importance"] is not None:
        print("\nNative modality ablation before CBP:")
        display(pd.DataFrame([result["modality_importance"]]).round(6))

    print("\nTop image patches:")
    display(result["patch_df"][[
        "y0","y1","x0","x1","importance","base_confidence","occluded_confidence"
    ]].head(12))

    if len(result["word_df"]) > 0:
        print("\nTop report words:")
        display(result["word_df"][[
            "word","positions","importance","base_confidence","occluded_confidence"
        ]].head(15))

    plot_result(result)


In [ ]:
# Verify explanation faithfulness by comparing model attributions against random-baseline perturbation curves, and provide a lookup wrapper to run verified explanations by patient UID.

def _curve_aucs(c):
    return float(c["deletion_auc"]), float(c["insertion_auc"])

@torch.inference_mode()
def verify_against_random(result, image, n_random=5, seed=SEED):
    """Re-run deletion/insertion with RANDOM ranking of the same number of units.
    An explanation is 'faithful vs random' if deleting its top units hurts confidence
    more (lower deletion AUC) and inserting them recovers it faster (higher insertion AUC)."""
    rng = np.random.default_rng(seed)
    pixel_values, input_ids, attention_mask = prepare_inputs(image, result["findings_used"])
    tc = result["target_class"]
    patch_df, word_df = result["patch_df"], result["word_df"]

    n_patch = int((patch_df["importance"] > 0).sum()) or len(patch_df)
    n_word = int((word_df["importance"] > 0).sum()) or len(word_df)

    def rand_patch_df(k):
        d = patch_df.sample(n=k, random_state=int(rng.integers(1_000_000_000))).reset_index(drop=True)
        d["importance"] = 1.0
        return d

    def rand_word_df(k):
        d = word_df.sample(n=k, random_state=int(rng.integers(1_000_000_000))).reset_index(drop=True)
        d["importance"] = 1.0
        return d

    rand = {"image": [], "text": [], "joint": []}
    for _ in range(n_random):
        rand["image"].append(_curve_aucs(image_faithfulness_curves(
            pixel_values, input_ids, attention_mask, tc, "multimodal", rand_patch_df(n_patch))))
        if len(word_df) > 0:
            rand["text"].append(_curve_aucs(text_faithfulness_curves(
                pixel_values, input_ids, attention_mask, tc, rand_word_df(n_word))))
            # joint: sample the same total number of units from the combined pool
            n_joint = int((word_df["importance"] > 0).sum() + (patch_df["importance"] > 0).sum()) \
                      or (len(word_df) + len(patch_df))
            pool = [("w", i) for i in range(len(word_df))] + [("p", i) for i in range(len(patch_df))]
            picks = rng.choice(len(pool), size=min(n_joint, len(pool)), replace=False)
            w_idx = [pool[j][1] for j in picks if pool[j][0] == "w"]
            p_idx = [pool[j][1] for j in picks if pool[j][0] == "p"]
            wd = word_df.iloc[w_idx].copy(); wd["importance"] = 1.0
            pdf = patch_df.iloc[p_idx].copy(); pdf["importance"] = 1.0
            rand["joint"].append(_curve_aucs(joint_faithfulness_curves(
                pixel_values, input_ids, attention_mask, tc, wd, pdf)))

    explained = {
        "image": (result["image_curves"]["deletion_auc"], result["image_curves"]["insertion_auc"]),
        "text": None if result["text_curves"] is None else
                (result["text_curves"]["deletion_auc"], result["text_curves"]["insertion_auc"]),
        "joint": None if result["joint_curves"] is None else
                 (result["joint_curves"]["deletion_auc"], result["joint_curves"]["insertion_auc"]),
    }
    out = {}
    for kind in ("image", "text", "joint"):
        if explained[kind] is None or not rand[kind]:
            continue
        r = np.asarray(rand[kind], dtype=float)
        ed, ei = float(explained[kind][0]), float(explained[kind][1])
        rd, ri = float(r[:, 0].mean()), float(r[:, 1].mean())
        out[kind] = {
            "explained_deletion_auc": ed, "random_deletion_auc_mean": rd,
            "explained_insertion_auc": ei, "random_insertion_auc_mean": ri,
            "deletion_better_than_random": bool(ed < rd),
            "insertion_better_than_random": bool(ei > ri),
            "n_random": int(n_random),
        }
    return out

def display_verification(result):
    ver = result.get("verification") or {}
    print("\n" + "=" * 88)
    print("VERIFICATION: Explainability ranking vs random-order baseline")
    print("=" * 88)
    if not ver:
        print("No verification available.")
        return
    rows = []
    for kind, v in ver.items():
        rows.append({
            "explained": kind,
            "deletion AUC (lower=better)": round(v["explained_deletion_auc"], 4),
            "random deletion AUC": round(v["random_deletion_auc_mean"], 4),
            "insertion AUC (higher=better)": round(v["explained_insertion_auc"], 4),
            "random insertion AUC": round(v["random_insertion_auc_mean"], 4),
            "beats random": bool(v["deletion_better_than_random"] and v["insertion_better_than_random"]),
        })
    display(pd.DataFrame(rows))
    passed = sum(r["beats random"] for r in rows)
    print(f"Explanation beats random ordering on {passed} of {len(rows)} checks.")
    print("Caveat: this shows the explanation reflects what the MODEL used; it does not show the "
          "prediction is clinically correct.")
    mi = result.get("modality_importance")
    if mi is not None:
        dominant = "text" if mi["text_modality_importance"] > mi["image_modality_importance"] else "image"
        print(f"Modality ablation: image importance={mi['image_modality_importance']:.4f}, "
              f"text importance={mi['text_modality_importance']:.4f} -> prediction leans on {dominant}.")

def explain_by_uid(uid, filename=None, n_random=5):
    row = lookup_by_uid(uid, filename)
    findings = row["findings"]
    if pd.isna(findings) or not str(findings).strip():
        raise ValueError(f"UID {_norm_uid(uid)} has empty findings; multimodal inference is not possible for it.")
    image_path = resolve_image_path(row["filename"])
    image = Image.open(image_path).convert("RGB")
    result = explain_one_image(image, row["filename"])      # matches the same row by filename
    if result["prediction"]["condition"] != "multimodal":
        raise RuntimeError("Expected multimodal inference but got " + result["prediction"]["condition"])
    result["uid"] = _norm_uid(uid)
    result["image_path"] = image_path
    result["verification"] = verify_against_random(result, image, n_random=n_random)
    return result


In [ ]:
# Construct an interactive widget interface to search chest X-ray filenames from Google Drive, trigger multimodal inference with explainability, and display verified results against random baselines.

search_box = widgets.Text(value="", placeholder="type part of a filename to search, e.g. 00012345",
                          description="Search:", style={"description_width": "70px"},
                          layout=widgets.Layout(width="520px"))
match_dropdown = widgets.Dropdown(options=[], description="File:",
                                  style={"description_width": "70px"},
                                  layout=widgets.Layout(width="600px"))
findings_output = widgets.Output()   # shows findings text as soon as a file is selected
run_button = widgets.Button(description="Predict + verify", button_style="primary",
                            icon="play", layout=widgets.Layout(width="240px"))
output = widgets.Output()
LAST_RESULT = None
MAX_MATCHES = 50

def refresh_matches(_=None):
    q = search_box.value.strip().lower()
    if not q:
        match_dropdown.options = []
        with findings_output:
            clear_output()
        return
    hits = [f for f in metadata_df["filename"].astype(str) if q in f.lower()][:MAX_MATCHES]
    match_dropdown.options = hits
    if hits:
        match_dropdown.value = hits[0]

search_box.observe(refresh_matches, names="value")

def find_row(filename):
    row = lookup_metadata(filename)
    if row is None:
        raise KeyError(f"'{filename}' was not found in {os.path.basename(METADATA_CSV_PATH)}.")
    return row

def show_findings(change):
    filename = change["new"]
    with findings_output:
        clear_output(wait=True)
        if not filename:
            return
        try:
            row = find_row(filename)
        except KeyError as exc:
            print(f"ERROR: {exc}")
            return
        findings = row["findings"]
        print(f"Matched UID : {_norm_uid(row['uid'])}")
        print(f"Filename    : {row['filename']}")
        print("Findings    :")
        if pd.isna(findings) or not str(findings).strip():
            print("  (empty — multimodal inference will not be possible for this file)")
        else:
            print(f"  {findings}")

match_dropdown.observe(show_findings, names="value")

def explain_from_drive(filename, n_random=5):
    row = find_row(filename)
    findings = row["findings"]
    if pd.isna(findings) or not str(findings).strip():
        raise ValueError(f"UID {_norm_uid(row['uid'])} has empty findings; multimodal inference is not possible.")
    image_path = resolve_image_path(row["filename"])
    image = Image.open(image_path).convert("RGB")
    result = explain_one_image(image, row["filename"])
    if result["prediction"]["condition"] != "multimodal":
        raise RuntimeError("Expected multimodal inference but got " + result["prediction"]["condition"])
    result["uid"] = _norm_uid(row["uid"])
    result["image_path"] = image_path
    result["verification"] = verify_against_random(result, image, n_random=n_random)
    return result

def run_clicked(_):
    global LAST_RESULT
    with output:
        clear_output(wait=True)
        try:
            if not match_dropdown.value:
                raise ValueError("Search for a filename and pick one from the dropdown first.")
            print("Loading from Drive, then running multimodal prediction + Explainability metrics + Heatmap generation...")
            LAST_RESULT = explain_from_drive(match_dropdown.value)
            clear_output(wait=True)
            print(f"Drive file  : {LAST_RESULT['image_path']}")
            print(f"Matched UID : {LAST_RESULT['uid']}")
            display_result(LAST_RESULT)
            display_verification(LAST_RESULT)
        except Exception as exc:
            print(f"ERROR: {type(exc).__name__}: {exc}")
            raise

run_button.on_click(run_clicked)

display(widgets.VBox([
    widgets.HTML("<h3>Search Drive CXR by filename → prediction → verified by Explainability</h3>"),
    search_box, match_dropdown, findings_output, run_button, output,
]))